# DBSCAN Example (Mall Customers Dataset)

**Goal: Find clusters of shoppers by Income and Spending Score without specifying the number of clusters.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/unsupervised/ at the repo root
SRC_UNSUP = os.path.join(REPO_ROOT, 'src', 'unsupervised')
sys.path.insert(0, SRC_UNSUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from dbscan import DBSCAN
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv(os.path.join(DATA_DIR, 'Mall_Customers.csv'))
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_raw = mall[MALL_FEATURES].values.astype(float)
X_mall = StandardScaler().fit_transform(X_raw)
print(f"Dataset loaded: {mall.shape[0]} samples.")

## 2. Run DBSCAN

Applied to the 2D slice of Annual Income and Spending Score (standardised).

In [ ]:
X_2feat = X_mall[:, 1:]
db = DBSCAN(eps=0.45, min_samples=4).fit(X_2feat)
print(f'Clusters found: {db.n_clusters_}')
print(f'Noise points:   {(db.labels_==-1).sum()}')
print(f'Core points:    {len(db.core_samples_)}')

## 3. Results and Visualisation

DBSCAN vs K-Means side-by-side on the same Income vs Spending Score slice.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_db = plt.cm.get_cmap('tab10', db.n_clusters_)

for lbl in np.unique(db.labels_):
    mask = db.labels_ == lbl
    if lbl == -1:
        axes[0].scatter(X_raw[mask,1], X_raw[mask,2], c='lightgray', s=20, marker='x', zorder=2, label='Noise')
    else:
        axes[0].scatter(X_raw[mask,1], X_raw[mask,2], color=cmap_db(lbl), s=35, alpha=0.8, label=f'Cluster {lbl}')
axes[0].scatter(X_raw[db.core_samples_,1], X_raw[db.core_samples_,2],
                edgecolors='black', facecolors='none', s=55, linewidths=0.8, zorder=3, label='Core')
axes[0].set_xlabel('Annual Income (k$)'); axes[0].set_ylabel('Spending Score (1-100)')
axes[0].set_title(f'DBSCAN - {db.n_clusters_} clusters found', fontweight='bold')
axes[0].legend(fontsize=7)

km2 = KMeans(k=5, init='k-means++', n_init=5, random_state=42).fit(X_2feat)
for c in range(5):
    mask = km2.labels_==c
    axes[1].scatter(X_raw[mask,1], X_raw[mask,2], color=plt.cm.tab10(c), s=30, alpha=0.75, label=f'C{c}')
axes[1].set_xlabel('Annual Income (k$)'); axes[1].set_ylabel('Spending Score (1-100)')
axes[1].set_title('K-Means (k=5) - Same 2D Slice', fontweight='bold'); axes[1].legend(fontsize=7)
fig.suptitle('DBSCAN vs K-Means on Income and Spending Score', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Analysis

**DBSCAN found 2 clusters, 9 noise points, 185 core points** on the Income vs Spending Score slice.

This is a much simpler partition than K-Means's 5 clusters on the same 2D slice. The reason is that DBSCAN groups points by density — and on this dataset, the overall density is fairly uniform. With eps=0.45 and min_samples=4, DBSCAN essentially finds one large dense region of average-spending customers and one region of high-spending customers, labelling the 9 most isolated points as noise.

**The 9 noise points** are the genuinely anomalous customers — those whose income/spending combination is unlike any of their neighbours. These outliers are valuable to identify: they might represent data entry errors, unusual shoppers, or genuinely rare customer types.

**185 out of 200 points are core points**, meaning most customers have at least 4 neighbours within distance 0.45 in standardised space. The data is dense enough that DBSCAN doesn't fragment into many small clusters.

**Comparison to K-Means on the same slice:** K-Means forces all 200 points into 5 equal-competition clusters regardless of density, identifying the well-known five spending segments (low income/high spend, high income/high spend, etc.). DBSCAN finds a coarser but density-justified partition. Neither is "wrong" — they answer different questions. K-Means finds the best k-way partition; DBSCAN finds natural density peaks.

**Key takeaway:** DBSCAN is most useful here for identifying the 9 outlier customers and confirming that the mall customer data is relatively uniformly dense. If the goal is detailed segmentation, K-Means with a well-chosen k is more appropriate for this particular dataset. DBSCAN shines on datasets with irregular cluster shapes and clear density gaps, which this one largely lacks.